# VAMP cross validation - code walkthrough with deeptime

Use VAMP-CV to optimize:

* feature set
* tICA lag
* tICA dimension
* number of clusters
* clustering algorithm

Then, for the best-performing representation:

1. build MSMs at multiple lag times;
2. inspect implied timescales;
3. run CK tests;
4. choose the smallest lag time where Markovian behavior is approximately achieved.


In [3]:
data_folder = "./data/"

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
binary_cmap = ListedColormap(['white', 'black'])

from tqdm.notebook import tqdm  
from deeptime.clustering import KMeans

from deeptime.decomposition import VAMP
import numpy as np


In [4]:
# set 1: distances
distances = np.loadtxt(data_folder + "hp35.mindists2", delimiter = " ", dtype = float)
print("Data shape", distances.shape) # [nm] nanometers
print("Data size (MB)", distances.nbytes / (10**6))

# set 2: dihedrals
#dihedrals = np.loadtxt(data_folder + "hp35.dihs.shifted", delimiter = " ", dtype = float)
#print(dihedrals.shape) # [radians] 

Data shape (1526041, 42)
Data size (MB) 512.749776


--- 
the manual way: define yourself the train/test, and the pipeline up to the clustering

In [5]:
def one_hot(dtraj, n_states):
    X = np.zeros((len(dtraj), n_states))
    X[np.arange(len(dtraj)), dtraj] = 1.0
    return X

In [6]:
length = distances.shape[0]
train = distances[:(length//2)]
test = distances[(length//2):]

In [11]:
## fit pca on training data only
pca = PCA(n_components=10)
Z_train = pca.fit_transform(train)
Z_test = pca.transform(test)


# clustering on the training data only
kmeans = KMeans(
    n_clusters=100,
    init_strategy='uniform',
    max_iter=100,
    fixed_seed=13,
    n_jobs=8,
    progress=tqdm
)


kmeans.fit(Z_train)


dtraj_train = kmeans.transform(Z_train)
dtraj_test = kmeans.transform(Z_test)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

/Users/miriamzara/miniconda3/envs/MSM_env/lib/python3.11/site-packages/deeptime/clustering/_kmeans.py:466: UserWarning: Algorithm did not reach convergence criterion of 1e-05 in 100 iterations. Consider increasing max_iter.
  warnings.warn(f"Algorithm did not reach convergence criterion"


In [17]:
# ------------------------
# one-hot encoding
# ------------------------

Z_train = one_hot(dtraj_train, 100)

Z_test = one_hot(dtraj_test, 100)


# ------------------------
# VAMP score
# ------------------------

from deeptime.decomposition import VAMP
scores = []
for time in np.arange(10, 100, 10):
    vamp = VAMP(lagtime=time)
    vamp_train_model = vamp.fit(Z_train).fetch_model()
    vamp_test_model = vamp.fit(Z_test).fetch_model()
    
    score = vamp_train_model.score(
        test_model = vamp_test_model,
        r=2,
    )
    scores.append(score)

While the estimator solves the problem of finding optimal transforms 
f
f, 
g
g, and 
K
K, one of the advantages of using the variational approach is that the models can be scored with respect to the choice of the system’s featurization.

This makes it possible to not only optimize for transforms given a basis but also for the basis itself






In addition to the score_method parameter, there is also a test_model parameter. By default it is None and the score is evaluated as-is. In case it is not None, the score() method evaluates the cross-validation score between the current model and test_model.

It is assumed that the model and the test_model stem from respective “training” and “testing” datasets. Estimation of the average cross-validation score and partitioning of data into test and training part is not performed by this method

In [18]:
scores

[np.float64(14.44686258641512),
 np.float64(11.170121964514006),
 np.float64(9.58824279116939),
 np.float64(8.590305340528474),
 np.float64(7.863911801146991),
 np.float64(7.3036875891349995),
 np.float64(6.883728255899129),
 np.float64(6.531132615148657),
 np.float64(6.209152999074909)]

---
the automated way: use vamp_score_cv and specify the number of kfolds

In [5]:
import numpy as np
from deeptime.decomposition import VAMP, vamp_score_cv, TICA
from deeptime.clustering import KMeans

lagtime = 10

def make_fit_fetch(n_tica, n_clusters, lagtime):
    def fit_fetch(trajs):
        # 1. PCA
        tica = TICA(lagtime=lagtime, dim=n_tica)
        tica.fit(trajs)
        tica_model = tica.fetch_model()

        # 2. Transform & cluster
        tica_output = [tica_model.transform(traj) for traj in trajs]
        km = KMeans(n_clusters=n_clusters, max_iter=100, progress=tqdm)
        km.fit(np.concatenate(tica_output))
        km_model = km.fetch_model()
        dtrajs = [km_model.transform(t) for t in tica_output]

        # 3. One-hot encode and fit VAMP directly
        def to_onehot(dtraj):
            oh = np.zeros((len(dtraj), n_clusters))
            oh[np.arange(len(dtraj)), dtraj] = 1.0
            return oh

        onehot_trajs = [to_onehot(d) for d in dtrajs]
        vamp = VAMP(lagtime=lagtime, dim=min(n_tica, n_clusters - 1))
        vamp.fit(onehot_trajs)
        return vamp.fetch_model()

    return fit_fetch

trajs = [distances]  # your raw feature trajectories

results = {}
for n_tica in [2]:
    for n_clusters in [20]:
        fit_fetch = make_fit_fetch(n_tica, n_clusters, lagtime)
        scores = vamp_score_cv(fit_fetch, trajs=trajs, blocksize=lagtime, n=5, r=2)
        results[(n_tica, n_clusters)] = scores.mean()
        print(f"TICA={n_tica}, clusters={n_clusters}: VAMP-2 = {scores.mean():.4f} ± {scores.std():.4f}")

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

/Users/miriamzara/miniconda3/envs/MSM_env/lib/python3.11/site-packages/deeptime/clustering/_kmeans.py:466: UserWarning: Algorithm did not reach convergence criterion of 1e-05 in 100 iterations. Consider increasing max_iter.
  warnings.warn(f"Algorithm did not reach convergence criterion"


...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

TICA=2, clusters=20: VAMP-2 = 1.4621 ± 0.0982


- k clusters == k basis functions

- each singular value satisfies $|\sigma_i| \leq 1$

- one singular value is exactly $\sigma_1 = 1$ (equilibrium distribution)

- the others are less

- VAMP SCORE satisfies

$$
0 \leq VAMP1 \leq k
$$

$$
0 \leq VAMP2 \leq k^2
$$

---
# Hyperparam selection with VAMP2 score, cross validated

Output: best model for each combination of dim.red. method + feature set. Total: 4 models.

I wont do an extensive grid search because of limited computing resources. 

### first optimize for cluster dimension (is the most cpu intensive step)

In [ ]:
output_folder = "intermediate_outputs/vamp_cv/"

feature_name = "distances"
vamp_lag = 500 # ~ 0.1 microseconds. this is an heuristic, but it should suffice to roughly select hyperparameters.
n_dim_list = [5] # intermediate dimension. this is also an heuristic at this stage.

n_clusters_list = [50, 100, 200]

In [ ]:
#############################

############################

"""
def make_fit_fetch(n_dim, lagtime, dimred_method = "tICA", n_clusters = 100):

    def fit_fetch(trajs):

        if dimred_method == "PCA":
            pca = PCAWrapper(dim=n_dim)
            pca.fit(np.concatenate(trajs))
            model = pca.fetch_model()

        if dimred_method == "tICA":
            tica = TICA(lagtime=lagtime, dim=n_dim)
            tica.fit(np.concatenate(trajs))
            model = tica.fetch_model()

        # 2. Transform & cluster
        dimred_output = [model.transform(traj) for traj in trajs]
        km = KMeans(n_clusters=n_clusters, max_iter=100, progress=tqdm, n_jobs = 8) #TODO fix njobs!!!
        km.fit(np.concatenate(dimred_output))
        km_model = km.fetch_model()
        dtrajs = [km_model.transform(t) for t in dimred_output]

        # 3. One-hot encode and fit VAMP directly
        def to_onehot(dtraj):
            oh = np.zeros((len(dtraj), n_clusters))
            oh[np.arange(len(dtraj)), dtraj] = 1.0
            return oh

        onehot_trajs = [to_onehot(d) for d in dtrajs]
        vamp = VAMP(lagtime=lagtime, dim=min(n_dim, n_clusters - 1))
        vamp.fit(onehot_trajs)
        return vamp.fetch_model()

    return fit_fetch

trajs = [distances]  # your raw feature trajectories




results = {}
for dimred_method in ["PCA", "tICA"]:
    for n_dim in n_dim_list:
        for n_clusters in n_clusters_list:
            fit_fetch = make_fit_fetch(n_dim = n_dim, dimred_method = dimred_method, n_clusters = n_clusters, lagtime = vamp_lag)
            scores = vamp_score_cv(fit_fetch, trajs=trajs, blocksize=vamp_lag*100, n=5, r=2)
            results[(n_dim, n_clusters, dimred_method, vamp_lag)] = scores.mean()
            print(f"{dimred_method}{n_dim}, clusters={n_clusters}: VAMP-2 = {scores.mean():.4f} ± {scores.std():.4f}")





rows = []
for (n_dim, n_clusters, dimred_method, vamp_lag), score in results.items():
    rows.append({
        "dimred_method": dimred_method,
        "n_clusters": n_clusters,
        "n_dim": n_dim,
        "lag_frames": vamp_lag,
        "mean_score": score[0],
        "std": score[1]
    })

df = pd.DataFrame(rows)

df.to_csv("intermediate_outputs/vamp_cv_nclusters.csv", index=False)
df.head()
"""